# 02 — Merge & Classify
Merges World Bank + OWID energy data, handles nulls, classifies demographic phases, and builds UN age-structure projections.
**Input:** `population_demographics.csv`, `energy_data.csv`, `population_projections.csv`, `population_age_projections.csv`
**Output:** `actuals_classified.csv`, `actuals_final.csv`, `model_df.csv`, `age_clean.csv`, `population_long_run.csv`

In [1]:
import pandas as pd
import numpy as np

COUNTRIES = {
    "JPN": "Japan", "DEU": "Germany", "CHN": "China", "NGA": "Nigeria",
    "IND": "India", "IDN": "Indonesia", "USA": "United States", "BRA": "Brazil",
}

pop_wb = pd.read_csv("population_demographics.csv")
energy_df = pd.read_csv("energy_data.csv")
proj_raw = pd.read_csv("population_projections.csv")
pop_age_proj = pd.read_csv("population_age_projections.csv")

print(f"World Bank: {pop_wb.shape}, Energy: {energy_df.shape}")

World Bank: (520, 11), Energy: (480, 130)


In [2]:
# Merge actuals
actuals = pop_wb.copy()
energy_renamed = energy_df.rename(columns={"iso_code": "country_code"})
energy_cols = [c for c in energy_renamed.columns if c not in ["country"]]
actuals = actuals.merge(energy_renamed[energy_cols], on=["country_code", "year"], how="left")
actuals = actuals.sort_values(["country_code", "year"]).reset_index(drop=True)
print(f"Actuals merged: {actuals.shape[0]} rows, {actuals.shape[1]} columns")

Actuals merged: 520 rows, 138 columns


In [3]:
# Demographic phase classification
def classify_phase(row):
    growth = row.get("population_growth_pct")
    pct_65 = row.get("pop_65_plus_pct")
    if pd.isna(growth) or pd.isna(pct_65):
        return "unknown"
    if growth < 0:
        return "declining"
    elif growth < 0.3 and pct_65 > 15:
        return "peaked_aging"
    elif pct_65 > 10:
        return "growing_aging"
    else:
        return "growing_young"

actuals["demographic_phase"] = actuals.apply(classify_phase, axis=1)
print("Phase distribution (2023):")
print(actuals[actuals["year"] == 2023].set_index("country_code")["demographic_phase"])
actuals.to_csv("actuals_classified.csv", index=False)

Phase distribution (2023):
country_code
BRA    growing_aging
CHN        declining
DEU     peaked_aging
IDN    growing_young
IND    growing_young
JPN        declining
NGA    growing_young
USA    growing_aging
Name: demographic_phase, dtype: object


In [4]:
# Select final columns
final_cols = [
    "country_code", "country", "year",
    "population_total", "population_growth_pct", "age_dependency_ratio",
    "pop_65_plus_pct", "pop_working_age_pct", "urban_pop_pct",
    "gdp_per_capita_usd", "gdp_growth_pct",
    "primary_energy_consumption", "energy_per_capita",
    "electricity_demand", "electricity_demand_per_capita",
    "fossil_share_energy", "low_carbon_share_energy",
    "renewables_share_energy", "energy_per_gdp",
    "demographic_phase",
]
actuals_final = actuals[final_cols].copy()
print(f"actuals_final shape: {actuals_final.shape}")
print("\nNull counts:")
print(actuals_final.isnull().sum())
actuals_final.to_csv("actuals_final.csv", index=False)

actuals_final shape: (520, 20)

Null counts:
country_code                       0
country                            0
year                               0
population_total                   0
population_growth_pct              8
age_dependency_ratio               0
pop_65_plus_pct                    0
pop_working_age_pct                0
urban_pop_pct                      0
gdp_per_capita_usd                 7
gdp_growth_pct                     8
primary_energy_consumption        56
energy_per_capita                 56
electricity_demand               311
electricity_demand_per_capita    311
fossil_share_energy              100
low_carbon_share_energy          100
renewables_share_energy          100
energy_per_gdp                    71
demographic_phase                  0
dtype: int64


In [5]:
# Build lag features for modeling
model_df = actuals_final.sort_values(["country_code", "year"]).reset_index(drop=True)
features_to_lag = [
    "population_total", "population_growth_pct", "pop_65_plus_pct",
    "pop_working_age_pct", "urban_pop_pct", "gdp_per_capita_usd", "gdp_growth_pct",
]
for col in features_to_lag:
    model_df[f"{col}_lag1"] = model_df.groupby("country_code")[col].shift(1)
model_df.to_csv("model_df.csv", index=False)
print(f"model_df saved: {model_df.shape}")

model_df saved: (520, 27)


In [6]:
# Build UN age-structure projections
age_raw = pop_age_proj.copy()
age_raw = age_raw[age_raw["Code"].isin(COUNTRIES.keys())]
age_raw = age_raw[(age_raw["Year"] >= 1960) & (age_raw["Year"] <= 2100)]

age_raw["total_pop"] = age_raw["Total (Projected)"].combine_first(age_raw["Total"])
age_raw["age_65_plus"] = age_raw["Ages 65+ (Projected)"].combine_first(age_raw["Ages 65+"])
age_raw["under_25"] = age_raw["Under-25s (Projected)"].combine_first(age_raw["Under-25s"])
age_raw["under_15"] = age_raw["Under-15s (Projected)"].combine_first(age_raw["Under-15s"])
age_raw["age_25_64"] = age_raw["Ages 25-64 (Projected)"].combine_first(age_raw["Ages 25-64"])
age_raw["age_15_24"] = age_raw["under_25"] - age_raw["under_15"]
age_raw["age_15_64"] = age_raw["age_15_24"] + age_raw["age_25_64"]
age_raw["is_projected"] = age_raw["Total (Projected)"].notna()
age_raw["pop_65_plus_pct"] = (age_raw["age_65_plus"] / age_raw["total_pop"]) * 100
age_raw["pop_working_age_pct"] = (age_raw["age_15_64"] / age_raw["total_pop"]) * 100

age_clean = age_raw.rename(columns={"Code": "country_code", "Entity": "country", "Year": "year"})
age_clean = age_clean[["country_code", "country", "year", "total_pop",
                        "pop_65_plus_pct", "pop_working_age_pct", "is_projected"]]

# Calibrate UN values to match World Bank at 2024 handoff
offset_year = 2024
offsets = {}
for code in COUNTRIES.keys():
    wb_val = actuals_final[(actuals_final["country_code"] == code) &
                           (actuals_final["year"] == offset_year)]["pop_65_plus_pct"].values
    un_val = age_clean[(age_clean["country_code"] == code) &
                       (age_clean["year"] == offset_year)]["pop_65_plus_pct"].values
    if len(wb_val) > 0 and len(un_val) > 0:
        offsets[code] = wb_val[0] - un_val[0]

age_clean["pop_65_plus_pct_calibrated"] = age_clean.apply(
    lambda row: row["pop_65_plus_pct"] + offsets.get(row["country_code"], 0)
    if row["year"] > offset_year else row["pop_65_plus_pct"], axis=1)

age_clean.to_csv("age_clean.csv", index=False)
print(f"age_clean saved: {age_clean.shape}")
print("Calibration offsets (pp):", {k: round(v, 3) for k, v in offsets.items()})

age_clean saved: (1128, 8)
Calibration offsets (pp): {'JPN': np.float64(0.098), 'DEU': np.float64(0.022), 'CHN': np.float64(0.003), 'NGA': np.float64(0.0), 'IND': np.float64(0.002), 'IDN': np.float64(0.001), 'USA': np.float64(0.02), 'BRA': np.float64(0.004)}


In [7]:
# Long-run population (1960-2100)
proj_clean = proj_raw[proj_raw["Code"].isin(COUNTRIES.keys())].copy()
proj_clean = proj_clean[(proj_clean["Year"] >= 1960) & (proj_clean["Year"] <= 2100)]
proj_clean["population"] = proj_clean["Population (projections) (Projected)"].combine_first(
    proj_clean["Population"])
proj_clean["is_projected"] = proj_clean["Population (projections) (Projected)"].notna()
proj_clean = proj_clean.rename(columns={"Code": "country_code", "Entity": "country", "Year": "year"})
proj_clean = proj_clean[["country_code", "country", "year", "population", "is_projected"]]
latest_phase = actuals[actuals["year"] == 2023].set_index("country_code")["demographic_phase"]
proj_clean["phase_as_of_2023"] = proj_clean["country_code"].map(latest_phase)
proj_clean.to_csv("population_long_run.csv", index=False)
print(f"population_long_run saved: {proj_clean.shape}")

population_long_run saved: (1128, 6)
